# Native periodic Laplacian block encodings
Retain the original 32-point and 2-by-4 grids, depth objective, and seven/six physical bits. Full complex-state and unnormalized-block checks precede the original phase-aligned display and normalized fidelity. The circuit viewer is replaced with native drawings.

## Specify a block encoding, then inspect its scale

The editable one-dimensional root composes the shared Fourier-shift Laplacian
block. FlagQuantum constructs the quantum phase and transform operations.
The periodic boundary, grid dimensions and normalization are supplied by the
model; the compiler does not infer these from a differential equation.
Both the 32-point and 2-by-4 grids remain in the complete workflow.

Try interpreting the unnormalized projected column before the phase-aligned
display. Normalized fidelity can hide an incorrect scale or global phase, so
full-state and absolute-block checks are retained. The selector probability
is an algorithm output, not proof that all selector bits were cleaned.


In [ ]:
import json

import numpy as np

from examples.synthesis.community.paper_implementation_project.block_encoding_nd_laplacian.overview.model import (
    laplacian_1d,
    laplacian_nd,
)
from examples.synthesis.community.paper_implementation_project.block_encoding_nd_laplacian.overview.workflow import (
    run_case,
)
from flagquantum.drawer import draw

from flagquantum import synth
from flagquantum.synth import Output, QArray, QBit, QNum, allocate, qfunc
from examples.synthesis.community.paper_implementation_project.block_encoding_nd_laplacian.periodic_laplacian_1d.main import (
    laplacian_block,
)
from examples.synthesis.community.paper_implementation_project.block_encoding_nd_laplacian.periodic_laplacian_1d.comparisons.circuit import (
    reference_circuit,
)
from examples.synthesis.compare_states import compare_states


@qfunc
def periodic_laplacian(position: Output[QNum], selector: Output[QArray[QBit]]) -> None:
    allocate(5, position)
    allocate(2, selector)
    laplacian_block(position, selector)


preview = synth.synthesize(
    periodic_laplacian,
    objective="depth",
    max_qubits=7,
    max_workspace=0,
    max_operations=10000,
    memory_limit_bytes=1024**2,
)
compare_states(preview, reference_circuit(), ("position", "selector"))


In [ ]:
lap_scaled_1d = laplacian_1d(32)
assert lap_scaled_1d.shape == (32, 32)


In [ ]:
one_dimensional = run_case()
print("32-point grid resources:", one_dimensional.resources)
print(draw(one_dimensional.program.circuit, show_all_qubits=True))
print("Actual complex projected block:", one_dimensional.projected)
print("Source phase-aligned display:", one_dimensional.source_display)
print("Fresh theoretical column:", lap_scaled_1d[:, 0])


In [ ]:
print("Source normalized fidelity:", one_dimensional.fidelity)
assert one_dimensional.full_state_error < 1e-11
assert one_dimensional.block_error < 1e-11
assert np.isclose(one_dimensional.fidelity, 1.0)
print("Actual projection probability:", one_dimensional.projection_probability)


In [ ]:
lap_scaled_nd = laplacian_nd((2, 4))
assert lap_scaled_nd.shape == (8, 8)


In [ ]:
two_dimensional = run_case(multidimensional=True)
print("2-by-4 grid resources:", two_dimensional.resources)
print(draw(two_dimensional.program.circuit, show_all_qubits=True))
print("Actual complex projected block:", two_dimensional.projected)
print("Source phase-aligned display:", two_dimensional.source_display)
print("Fresh theoretical column:", lap_scaled_nd[:, 0])


In [ ]:
print("Source normalized fidelity:", two_dimensional.fidelity)
assert two_dimensional.full_state_error < 1e-11
assert two_dimensional.block_error < 1e-11
assert np.isclose(two_dimensional.fidelity, 1.0)
print("Actual projection probability:", two_dimensional.projection_probability)
report = {
    "grids": [[32], [2, 4]],
    "objective": "depth",
    "widths": [one_dimensional.resources.n_qubits, two_dimensional.resources.n_qubits],
    "operations": [
        one_dimensional.resources.n_operations,
        two_dimensional.resources.n_operations,
    ],
    "depths": [one_dimensional.resources.depth, two_dimensional.resources.depth],
    "state_errors": [
        one_dimensional.full_state_error,
        two_dimensional.full_state_error,
    ],
    "block_errors": [one_dimensional.block_error, two_dimensional.block_error],
    "projection_probabilities": [
        one_dimensional.projection_probability,
        two_dimensional.projection_probability,
    ],
    "fidelities": [one_dimensional.fidelity, two_dimensional.fidelity],
    "projected_real": [
        one_dimensional.projected.real.tolist(),
        two_dimensional.projected.real.tolist(),
    ],
    "projected_imag": [
        one_dimensional.projected.imag.tolist(),
        two_dimensional.projected.imag.tolist(),
    ],
}
print("ARTIFACT_JSON=" + json.dumps(report))
